# Metaclasses vs Class Decorators
## Advanced Tutorial Problems with Full Solutions — Volume 2

This notebook is intentionally written in a **tutorial style**.

Instead of presenting one large solution immediately, each problem is broken into small steps:

- first we state the goal,
- then we build a minimal experiment,
- then we inspect what Python actually did,
- then we explain the result,
- and finally we build the complete solution.

The central comparison is:

> A **class decorator** transforms a specific class after that class has already been created.

> A **metaclass** participates in the class-creation process itself, so its rules naturally continue when subclasses are created.

We will keep returning to that distinction from different angles.

## Topics covered

This second problem set explores:

1. inheritance across several generations,
2. `__init_subclass__` as a middle ground,
3. metaclass `__prepare__`,
4. duplicate-name detection in class bodies,
5. descriptor-safe transformation,
6. class-creation lifecycle order,
7. inheritance-wide validation,
8. automatic registries,
9. metaclass conflicts and cooperative composition,
10. compatibility with `ABCMeta`,
11. metaclass `__call__`,
12. descriptors and `__set_name__`,
13. a capstone plugin framework.

All intentional failure cases are caught so that the notebook can be executed from top to bottom.

In [1]:
from __future__ import annotations

from abc import ABC, ABCMeta, abstractmethod
from functools import wraps
import inspect
import itertools

print("Setup complete.")

Setup complete.


# Problem 1 — Follow a class decorator through a deeper inheritance tree

We begin with a familiar requirement:

> "Every method in this class family should be traced."

We first solve it with a class decorator.

The interesting question is not whether the root class works.

The interesting question is what happens to methods introduced by **future subclasses**.

## Step 1 — Build a function wrapper

The wrapper prints the qualified name of the original function and then returns its result.

We use `functools.wraps` so that important metadata from the original function is preserved.

In [2]:
def trace_function(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"TRACE {fn.__qualname__} -> {result!r}")
        return result
    return wrapper

## Step 2 — Turn it into a class decorator

We inspect `vars(cls)` because we only want attributes defined **directly on that class**.

In [3]:
def trace_class(cls):
    for name, obj in list(vars(cls).items()):
        if inspect.isfunction(obj):
            setattr(cls, name, trace_function(obj))
    return cls

## Step 3 — Decorate only the root class

We create three generations:

- `P1Root`
- `P1Middle`
- `P1Leaf`

Only the root class gets the decorator.

In [4]:
@trace_class
class P1Root:
    def root_method(self):
        return "root"


class P1Middle(P1Root):
    def middle_method(self):
        return "middle"


class P1Leaf(P1Middle):
    def leaf_method(self):
        return "leaf"

Before running the next cell, predict the result.

`root_method` was already wrapped before either subclass existed.

But did the decorator ever get a chance to see `middle_method` or `leaf_method`?

In [5]:
p1 = P1Leaf()

assert p1.root_method() == "root"
assert p1.middle_method() == "middle"
assert p1.leaf_method() == "leaf"

print("root wrapped:", hasattr(P1Root.root_method, "__wrapped__"))
print("middle wrapped:", hasattr(P1Middle.middle_method, "__wrapped__"))
print("leaf wrapped:", hasattr(P1Leaf.leaf_method, "__wrapped__"))

assert hasattr(P1Root.root_method, "__wrapped__")
assert not hasattr(P1Middle.middle_method, "__wrapped__")
assert not hasattr(P1Leaf.leaf_method, "__wrapped__")

TRACE P1Root.root_method -> 'root'
root wrapped: True
middle wrapped: False
leaf wrapped: False


## Explanation

The inherited method remains wrapped because the wrapped function itself is inherited.

But the class decorator does **not** run again when subclasses are created.

So methods added by future subclasses are invisible to the original decorator call.

## Step 4 — Solve the same problem with a metaclass

Now we move the transformation into metaclass `__new__`.

That hook runs each time a class using the metaclass is created.

In [6]:
class TraceMeta(type):
    def __new__(mcls, name, bases, namespace):
        new_namespace = dict(namespace)

        for attr_name, obj in namespace.items():
            if inspect.isfunction(obj):
                new_namespace[attr_name] = trace_function(obj)

        return super().__new__(mcls, name, bases, new_namespace)

In [7]:
class P1MetaRoot(metaclass=TraceMeta):
    def root_method(self):
        return "root"


class P1MetaMiddle(P1MetaRoot):
    def middle_method(self):
        return "middle"


class P1MetaLeaf(P1MetaMiddle):
    def leaf_method(self):
        return "leaf"

Notice that neither subclass explicitly mentions `TraceMeta`.

The metaclass choice is inherited through the class hierarchy.

In [8]:
p1m = P1MetaLeaf()

assert p1m.root_method() == "root"
assert p1m.middle_method() == "middle"
assert p1m.leaf_method() == "leaf"

assert hasattr(P1MetaRoot.root_method, "__wrapped__")
assert hasattr(P1MetaMiddle.middle_method, "__wrapped__")
assert hasattr(P1MetaLeaf.leaf_method, "__wrapped__")

print(type(P1MetaRoot).__name__)
print(type(P1MetaMiddle).__name__)
print(type(P1MetaLeaf).__name__)

TRACE P1MetaRoot.root_method -> 'root'
TRACE P1MetaMiddle.middle_method -> 'middle'
TRACE P1MetaLeaf.leaf_method -> 'leaf'
TraceMeta
TraceMeta
TraceMeta


### Problem 1 takeaway

A class decorator is excellent for explicit, local transformation.

A metaclass is useful when the transformation is part of the **class family's creation rules**.

# Problem 2 — Achieve inheritance propagation without a custom metaclass

A metaclass is not always necessary.

Python gives classes another hook:

```python
__init_subclass__
```

It runs whenever a subclass is created.

This makes it a very useful middle ground between a one-time class decorator and a full metaclass.

## Step 1 — Put the transformation in a base class

In [9]:
class AutoTraceChildren:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        for name, obj in list(vars(cls).items()):
            if inspect.isfunction(obj):
                setattr(cls, name, trace_function(obj))

## Step 2 — Create a hierarchy normally

No class decorator is repeated.

No custom metaclass is declared.

In [10]:
class P2Base(AutoTraceChildren):
    def one(self):
        return 1


class P2Child(P2Base):
    def two(self):
        return 2


class P2GrandChild(P2Child):
    def three(self):
        return 3

In [11]:
p2 = P2GrandChild()

assert p2.one() == 1
assert p2.two() == 2
assert p2.three() == 3

assert hasattr(P2Base.one, "__wrapped__")
assert hasattr(P2Child.two, "__wrapped__")
assert hasattr(P2GrandChild.three, "__wrapped__")

TRACE P2Base.one -> 1
TRACE P2Child.two -> 2
TRACE P2GrandChild.three -> 3


## When is `__init_subclass__` enough?

It is often preferable when:

- you control the base class,
- your rule only needs to run after subclass creation,
- you do not need `__prepare__`,
- you do not need to customize instance construction,
- and you want to avoid metaclass-composition problems.

This is an important best practice:

> use the least powerful hook that cleanly solves the problem.

# Problem 3 — Use `__prepare__` to observe the class body before the class exists

A class decorator receives a class **after** the class object exists.

A metaclass can intervene earlier.

The earliest common hook is `__prepare__`.

It controls the mapping used as the temporary namespace while the class body executes.

## Goal

We want Python to reject a class body that defines the same non-dunder attribute twice.

Normally, the second definition silently replaces the first.

## Step 1 — Create a namespace that rejects duplicate assignments

In [12]:
class DuplicateCheckingDict(dict):
    def __setitem__(self, key, value):
        if key in self and not key.startswith("__"):
            raise TypeError(f"duplicate class attribute: {key!r}")
        super().__setitem__(key, value)

## Step 2 — Return that namespace from `__prepare__`

In [13]:
class NoDuplicateMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return DuplicateCheckingDict()

    def __new__(mcls, name, bases, namespace, **kwargs):
        return super().__new__(mcls, name, bases, dict(namespace))

## Step 3 — A valid class still works normally

In [14]:
class P3Valid(metaclass=NoDuplicateMeta):
    x = 10
    y = 20

    def total(self):
        return self.x + self.y


assert P3Valid().total() == 30

## Step 4 — A duplicate definition fails during class-body execution

In [15]:
try:
    class P3Broken(metaclass=NoDuplicateMeta):
        def process(self):
            return "first"

        def process(self):
            return "second"
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Duplicate method definition should fail.")

Expected: duplicate class attribute: 'process'


## Why a class decorator cannot perfectly reproduce this

By the time a decorator receives the class, the first `process` function has already been overwritten.

The completed class contains only the second definition.

So this is a real example where **earlier access to the class-creation pipeline** matters.

# Problem 4 — Transform methods without breaking descriptors

A naïve implementation often checks only:

```python
inspect.isfunction(obj)
```

That handles ordinary methods, but class namespaces also contain descriptors such as:

- `staticmethod`
- `classmethod`
- `property`

We need to transform the underlying functions while preserving the descriptor objects.

## Step 1 — Inspect the raw class namespace

Attribute lookup can hide descriptor details.

`vars(SomeClass)` shows what is actually stored.

In [16]:
class P4Inspect:
    def normal(self):
        return "normal"

    @staticmethod
    def static():
        return "static"

    @classmethod
    def class_method(cls):
        return cls.__name__

    @property
    def value(self):
        return 42


for name in ["normal", "static", "class_method", "value"]:
    raw = vars(P4Inspect)[name]
    print(name, "->", type(raw).__name__, "callable?", callable(raw))

normal -> function callable? True
static -> staticmethod callable? True
class_method -> classmethod callable? False
value -> property callable? False


## Step 2 — Write a descriptor-aware transformation function

In [17]:
def marked_trace(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"CALL {fn.__qualname__}")
        return result

    wrapper.__traced__ = True
    return wrapper


def transform_descriptor(obj):
    if inspect.isfunction(obj):
        return marked_trace(obj)

    if isinstance(obj, staticmethod):
        return staticmethod(marked_trace(obj.__func__))

    if isinstance(obj, classmethod):
        return classmethod(marked_trace(obj.__func__))

    if isinstance(obj, property):
        return property(
            marked_trace(obj.fget) if obj.fget else None,
            marked_trace(obj.fset) if obj.fset else None,
            marked_trace(obj.fdel) if obj.fdel else None,
            doc=obj.__doc__,
        )

    return obj

## Step 3 — Use the transformer from a metaclass

In [18]:
class DescriptorTraceMeta(type):
    def __new__(mcls, name, bases, namespace):
        transformed = {
            key: transform_descriptor(value)
            for key, value in namespace.items()
        }
        return super().__new__(mcls, name, bases, transformed)

## Step 4 — Test every supported descriptor type

In [19]:
class P4Example(metaclass=DescriptorTraceMeta):
    factor = 5

    def __init__(self, value):
        self._value = value

    def normal(self, n):
        return self._value + n

    @staticmethod
    def add(a, b):
        return a + b

    @classmethod
    def scale(cls, n):
        return cls.factor * n

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        self._value = new_value


p4 = P4Example(10)

assert p4.normal(2) == 12
assert P4Example.add(2, 3) == 5
assert P4Example.scale(4) == 20
assert p4.value == 10

p4.value = 99
assert p4.value == 99

CALL P4Example.__init__
CALL P4Example.normal
CALL P4Example.add
CALL P4Example.scale
CALL P4Example.value
CALL P4Example.value
CALL P4Example.value


## Step 5 — Verify that we did not destroy descriptor types

In [20]:
raw = vars(P4Example)

assert isinstance(raw["add"], staticmethod)
assert isinstance(raw["scale"], classmethod)
assert isinstance(raw["value"], property)

print("Descriptor types preserved.")

Descriptor types preserved.


### Problem 4 takeaway

When doing metaprogramming, always distinguish:

- the result of normal attribute access,
- from the raw object stored in the class dictionary.

That distinction is especially important for descriptors.

# Problem 5 — Determine the exact order of class-creation events

We will now record the order of:

1. metaclass `__prepare__`,
2. class-body execution,
3. metaclass `__new__`,
4. metaclass `__init__`,
5. class decorators.

This is easier to remember after seeing it happen.

In [21]:
P5_EVENTS = []


def record_decorator(cls):
    P5_EVENTS.append(("decorator", cls.__name__))
    return cls


class LifecycleMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases):
        P5_EVENTS.append(("meta.__prepare__", name))
        return {}

    def __new__(mcls, name, bases, namespace):
        P5_EVENTS.append(("meta.__new__", name))
        return super().__new__(mcls, name, bases, namespace)

    def __init__(cls, name, bases, namespace):
        P5_EVENTS.append(("meta.__init__", name))
        super().__init__(name, bases, namespace)

## Step 1 — Put one recorded event directly in the class body

In [22]:
@record_decorator
class P5Example(metaclass=LifecycleMeta):
    P5_EVENTS.append(("class body", "P5Example"))

    def method(self):
        return "ok"

## Step 2 — Print the order

In [23]:
for event in P5_EVENTS:
    print(event)

('meta.__prepare__', 'P5Example')
('class body', 'P5Example')
('meta.__new__', 'P5Example')
('meta.__init__', 'P5Example')
('decorator', 'P5Example')


## Step 3 — Assert the sequence

In [24]:
assert P5_EVENTS == [
    ("meta.__prepare__", "P5Example"),
    ("class body", "P5Example"),
    ("meta.__new__", "P5Example"),
    ("meta.__init__", "P5Example"),
    ("decorator", "P5Example"),
]

## Explanation

The decorator is last.

That means the decorator cannot influence how the original class body executed.

It only sees the finished class after the metaclass has already created and initialized it.

# Problem 6 — Enforce a class contract across an inheritance tree

Suppose a framework requires every handler to have:

- a non-empty string `route`,
- a callable `handle`.

We will first validate with a class decorator, then expose its inheritance weakness.

## Step 1 — A class decorator can validate one class very cleanly

In [25]:
def validate_handler(cls):
    route = getattr(cls, "route", None)

    if not isinstance(route, str) or not route:
        raise TypeError(f"{cls.__name__}: invalid route")

    if not callable(getattr(cls, "handle", None)):
        raise TypeError(f"{cls.__name__}: missing handle()")

    return cls

In [26]:
@validate_handler
class P6DecoratedHandler:
    route = "/home"

    def handle(self):
        return "home"


assert P6DecoratedHandler().handle() == "home"

## Step 2 — A subclass can violate the rule if we forget the decorator

The parent was valid.

The child deliberately overrides the route with an invalid value.

In [27]:
class P6ForgottenChild(P6DecoratedHandler):
    route = ""

    def handle(self):
        return "still created"


assert P6ForgottenChild.route == ""
print("The invalid subclass still exists.")

The invalid subclass still exists.


## Step 3 — Put the invariant in a metaclass

Now every future subclass is checked automatically.

In [28]:
class HandlerMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if namespace.get("_framework_root", False):
            return cls

        route = getattr(cls, "route", None)

        if not isinstance(route, str) or not route:
            raise TypeError(f"{name}: invalid route")

        if not callable(getattr(cls, "handle", None)):
            raise TypeError(f"{name}: missing handle()")

        return cls

In [29]:
class P6HandlerRoot(metaclass=HandlerMeta):
    _framework_root = True


class P6GoodHandler(P6HandlerRoot):
    route = "/good"

    def handle(self):
        return "good"


assert P6GoodHandler().handle() == "good"

## Step 4 — The invalid subclass now fails during class creation

In [30]:
try:
    class P6BadHandler(P6GoodHandler):
        route = ""
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Invalid subclass should not be created.")

Expected: P6BadHandler: invalid route


### Problem 6 takeaway

A decorator is excellent for a class-level check that is intentionally opt-in.

A metaclass is appropriate when the rule is a framework invariant that must automatically continue through inheritance.

# Problem 7 — Build an automatic registry

Registries are common in serializers, commands, plugins, ORM systems, and dispatch frameworks.

We want subclasses to register themselves by a string key.

## Step 1 — The explicit decorator version

In [31]:
P7_DECORATOR_REGISTRY = {}


def register(key):
    def decorator(cls):
        if key in P7_DECORATOR_REGISTRY:
            raise KeyError(f"duplicate key: {key}")
        P7_DECORATOR_REGISTRY[key] = cls
        return cls
    return decorator

In [32]:
@register("csv")
class P7Csv:
    pass


@register("json")
class P7Json:
    pass


assert P7_DECORATOR_REGISTRY["csv"] is P7Csv
assert P7_DECORATOR_REGISTRY["json"] is P7Json

This is explicit and readable.

But registration must be repeated on every class that should participate.

## Step 2 — Automatic registration with a metaclass

In [33]:
class RegistryMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        key = namespace.get("registry_key")

        if key is not None:
            if key in mcls.registry:
                raise KeyError(f"duplicate key: {key}")
            mcls.registry[key] = cls

        return cls

In [34]:
class P7Base(metaclass=RegistryMeta):
    pass


class P7Yaml(P7Base):
    registry_key = "yaml"


class P7Toml(P7Base):
    registry_key = "toml"


assert RegistryMeta.registry["yaml"] is P7Yaml
assert RegistryMeta.registry["toml"] is P7Toml

## Step 3 — Enforce uniqueness automatically

In [35]:
try:
    class P7Duplicate(P7Base):
        registry_key = "yaml"
except KeyError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Duplicate registry key should fail.")

Expected: 'duplicate key: yaml'


## Which version is better?

The decorator version is often better when registration is optional and should be visually obvious.

The metaclass version is often better when registration is part of the framework's class-creation contract.

# Problem 8 — Diagnose and repair a metaclass conflict

One disadvantage of metaclasses appears with multiple inheritance.

If two base classes use incompatible custom metaclasses, Python cannot automatically choose one metaclass for the derived class.

In [36]:
class MetaLeft(type):
    pass


class MetaRight(type):
    pass


class P8Left(metaclass=MetaLeft):
    pass


class P8Right(metaclass=MetaRight):
    pass

## Step 1 — Trigger the conflict intentionally

In [37]:
try:
    class P8Broken(P8Left, P8Right):
        pass
except TypeError as exc:
    print("Expected conflict:")
    print(exc)
else:
    raise AssertionError("A metaclass conflict was expected.")

Expected conflict:
metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases


## Step 2 — Build a composite metaclass

The new metaclass is a subclass of both required metaclasses.

In [38]:
class MetaCombined(MetaLeft, MetaRight):
    pass


class P8Fixed(P8Left, P8Right, metaclass=MetaCombined):
    pass

In [39]:
assert isinstance(P8Fixed, MetaLeft)
assert isinstance(P8Fixed, MetaRight)

print("chosen metaclass:", type(P8Fixed).__name__)
print("metaclass MRO:", [c.__name__ for c in MetaCombined.__mro__])

chosen metaclass: MetaCombined
metaclass MRO: ['MetaCombined', 'MetaLeft', 'MetaRight', 'type', 'object']


## Step 3 — Cooperative metaclasses should use `super()`

Composition becomes much safer when each metaclass participates in the metaclass MRO instead of directly calling `type.__new__`.

In [40]:
P8_LOG = []


class CooperativeLeft(type):
    def __new__(mcls, name, bases, namespace):
        P8_LOG.append(("left", name))
        return super().__new__(mcls, name, bases, namespace)


class CooperativeRight(type):
    def __new__(mcls, name, bases, namespace):
        P8_LOG.append(("right", name))
        return super().__new__(mcls, name, bases, namespace)


class CooperativeCombined(CooperativeLeft, CooperativeRight):
    pass


class P8Cooperative(metaclass=CooperativeCombined):
    pass


assert P8_LOG == [
    ("left", "P8Cooperative"),
    ("right", "P8Cooperative"),
]

print(P8_LOG)

[('left', 'P8Cooperative'), ('right', 'P8Cooperative')]


### Problem 8 takeaway

The "one metaclass" rule is more subtle than it first appears.

A derived class can use a metaclass that is compatible with all base metaclasses.

That usually means creating a cooperative composite metaclass.

# Problem 9 — Combine custom metaclass behavior with `ABCMeta`

Abstract base classes use `ABCMeta`.

So if we want custom metaclass behavior and abstract methods at the same time, our metaclass should normally derive from `ABCMeta`.

## Step 1 — Build a simple call-counting function wrapper

In [41]:
def counted(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)

    wrapper.calls = 0
    return wrapper

## Step 2 — Build a metaclass that extends `ABCMeta`

In [42]:
class CountingABCMeta(ABCMeta):
    def __new__(mcls, name, bases, namespace):
        new_namespace = dict(namespace)

        for key, value in namespace.items():
            if inspect.isfunction(value):
                new_namespace[key] = counted(value)

        return super().__new__(mcls, name, bases, new_namespace)

## Step 3 — Define an abstract base class

`functools.wraps` preserves the wrapped function's metadata dictionary, including the abstract-method marker.

In [43]:
class P9Processor(ABC, metaclass=CountingABCMeta):
    @abstractmethod
    def process(self, value):
        raise NotImplementedError


assert inspect.isabstract(P9Processor)

Trying to instantiate the abstract root must still fail.

In [44]:
try:
    P9Processor()
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Abstract class should not be instantiable.")

Expected: Can't instantiate abstract class P9Processor without an implementation for abstract method 'process'


## Step 4 — A concrete subclass is automatically processed by the same metaclass

In [45]:
class P9Double(P9Processor):
    def process(self, value):
        return value * 2


p9 = P9Double()

assert p9.process(10) == 20
assert P9Double.process.calls == 1
assert not inspect.isabstract(P9Double)

print("calls:", P9Double.process.calls)

calls: 1


### Problem 9 takeaway

Classes have an inheritance hierarchy.

Metaclasses also have an inheritance hierarchy.

Advanced designs often require both hierarchies to cooperate.

# Problem 10 — Customize instance construction with metaclass `__call__`

When we write:

```python
SomeClass(...)
```

Python calls the metaclass's `__call__`.

That gives a metaclass control over instance creation without replacing the class object with a factory function.

## Goal

Implement per-class interning:

- same constructor arguments -> same object,
- different arguments -> different object,
- subclasses -> independent caches.

In [46]:
class InternMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls._instance_cache = {}
        return cls

    def __call__(cls, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))

        if key not in cls._instance_cache:
            cls._instance_cache[key] = super().__call__(*args, **kwargs)

        return cls._instance_cache[key]

## Step 1 — Create a class using the metaclass

In [47]:
class P10Token(metaclass=InternMeta):
    def __init__(self, name, level=0):
        self.name = name
        self.level = level

## Step 2 — Equal constructor arguments return the same instance

In [48]:
a = P10Token("admin", level=1)
b = P10Token("admin", level=1)
c = P10Token("guest", level=1)

assert a is b
assert a is not c

print("a is b:", a is b)
print("a is c:", a is c)

a is b: True
a is c: False


## Step 3 — Subclasses receive their own cache

In [49]:
class P10SpecialToken(P10Token):
    pass


parent = P10Token("x")
child = P10SpecialToken("x")

assert parent is not child
assert type(parent) is P10Token
assert type(child) is P10SpecialToken

## Why this is interesting

A class decorator could replace a class name with some callable factory.

But then the name might no longer refer to a normal class.

A metaclass can customize construction while preserving normal class identity, inheritance, and `isinstance` behavior.

# Problem 11 — Inject descriptors and understand `__set_name__`

Descriptors may define `__set_name__`.

Python automatically calls that method for descriptors that are present in the class namespace during class creation.

This creates a subtle timing difference between decorators and metaclasses.

## Step 1 — Build a descriptor that remembers its assigned name

In [50]:
class NamedField:
    def __init__(self, default=None):
        self.default = default
        self.public_name = None
        self.private_name = None

    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.private_name, self.default)

    def __set__(self, instance, value):
        setattr(instance, self.private_name, value)

## Step 2 — Add the descriptor after class creation with a decorator

The descriptor was not present when `type.__new__` automatically called `__set_name__`.

In [51]:
def add_field_late(cls):
    cls.extra = NamedField(default="late")
    return cls


@add_field_late
class P11Decorated:
    pass


print("public_name:", P11Decorated.extra.public_name)
assert P11Decorated.extra.public_name is None

public_name: None


## Step 3 — Repair the decorator manually

Because the descriptor was attached late, the decorator can explicitly call `__set_name__`.

In [52]:
def add_field_late_correctly(cls):
    field = NamedField(default="late-fixed")
    cls.extra = field
    field.__set_name__(cls, "extra")
    return cls


@add_field_late_correctly
class P11DecoratedFixed:
    pass


assert P11DecoratedFixed.extra.public_name == "extra"
assert P11DecoratedFixed().extra == "late-fixed"

## Step 4 — Inject the descriptor before `type.__new__`

A metaclass can put the descriptor into the namespace early enough for Python to perform the normal descriptor lifecycle automatically.

In [53]:
class FieldInjectingMeta(type):
    def __new__(mcls, name, bases, namespace):
        new_namespace = dict(namespace)
        new_namespace.setdefault("extra", NamedField(default="metaclass"))
        return super().__new__(mcls, name, bases, new_namespace)


class P11MetaExample(metaclass=FieldInjectingMeta):
    pass


assert P11MetaExample.extra.public_name == "extra"
assert P11MetaExample().extra == "metaclass"

print(P11MetaExample.extra.public_name)

extra


### Problem 11 takeaway

A decorator can still solve the problem, but because it runs later it may need to manually reproduce lifecycle work that Python normally performs during class creation.

A metaclass can often participate at the natural point in that lifecycle.

# Problem 12 — Capstone: a small plugin framework

We will combine several ideas into one design.

The framework should provide:

- an abstract plugin root,
- automatic registration of concrete subclasses,
- unique plugin names,
- automatic creation-order numbers,
- optional human-readable descriptions.

We will deliberately split responsibilities:

- the **metaclass** handles mandatory framework invariants,
- a **class decorator** handles optional local metadata.

## Step 1 — Optional metadata belongs nicely in a class decorator

In [54]:
def described(text):
    def decorator(cls):
        cls.description = text
        return cls
    return decorator

## Step 2 — Build the framework metaclass

Because the root class is abstract, the metaclass extends `ABCMeta`.

In [55]:
class PluginMeta(ABCMeta):
    registry = {}
    sequence = itertools.count(1)

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        cls.creation_order = next(mcls.sequence)

        if inspect.isabstract(cls):
            return cls

        plugin_name = getattr(cls, "plugin_name", None)

        if not isinstance(plugin_name, str) or not plugin_name:
            raise TypeError(
                f"{name}: concrete plugins need a non-empty string 'plugin_name'"
            )

        if plugin_name in mcls.registry:
            other = mcls.registry[plugin_name]
            raise TypeError(
                f"{name}: plugin name {plugin_name!r} already belongs to {other.__name__}"
            )

        mcls.registry[plugin_name] = cls
        return cls

## Step 3 — Define the abstract root

In [56]:
class P12Plugin(ABC, metaclass=PluginMeta):
    @abstractmethod
    def run(self, data):
        raise NotImplementedError


assert inspect.isabstract(P12Plugin)
assert P12Plugin not in PluginMeta.registry.values()

## Step 4 — Create concrete plugins

Only one plugin receives the optional description decorator.

In [57]:
@described("Converts text to upper case")
class P12Upper(P12Plugin):
    plugin_name = "upper"

    def run(self, data):
        return str(data).upper()


class P12Repeat(P12Plugin):
    plugin_name = "repeat"

    def run(self, data):
        return str(data) * 2

In [58]:
assert PluginMeta.registry["upper"] is P12Upper
assert PluginMeta.registry["repeat"] is P12Repeat

assert P12Upper().run("abc") == "ABC"
assert P12Repeat().run("ab") == "abab"

assert P12Upper.description == "Converts text to upper case"
assert not hasattr(P12Repeat, "description")

## Step 5 — Inspect automatic creation order

In [59]:
print("root:", P12Plugin.creation_order)
print("upper:", P12Upper.creation_order)
print("repeat:", P12Repeat.creation_order)

assert P12Plugin.creation_order < P12Upper.creation_order < P12Repeat.creation_order

root: 1
upper: 2
repeat: 3


## Step 6 — Duplicate names fail automatically

In [60]:
try:
    class P12Duplicate(P12Plugin):
        plugin_name = "upper"

        def run(self, data):
            return data
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Duplicate plugin name should fail.")

Expected: P12Duplicate: plugin name 'upper' already belongs to P12Upper


## Step 7 — Missing metadata also fails automatically

In [61]:
try:
    class P12MissingName(P12Plugin):
        def run(self, data):
            return data
except TypeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Missing plugin_name should fail.")

Expected: P12MissingName: concrete plugins need a non-empty string 'plugin_name'


## Step 8 — Use the registry as a factory

In [62]:
def make_plugin(name):
    plugin_cls = PluginMeta.registry[name]
    return plugin_cls()


plugin = make_plugin("upper")

assert plugin.run("hello") == "HELLO"
print(type(plugin).__name__)

P12Upper


# Final comparison

## Class decorators

A class decorator is a strong choice when:

- the behavior is local,
- opt-in behavior is desirable,
- explicit syntax helps readability,
- inheritance-wide enforcement is not required.

## `__init_subclass__`

This is a strong choice when:

- you control the base class,
- behavior should continue through subclasses,
- subclass initialization is enough,
- you want to avoid a custom metaclass.

## Metaclasses

A metaclass becomes useful when:

- behavior belongs to class creation itself,
- future subclasses must automatically obey the same rules,
- you need `__prepare__`,
- you need class-definition-time validation,
- you need automatic registries,
- you need to customize class calling through metaclass `__call__`,
- or you need to cooperate with another metaclass such as `ABCMeta`.

The general best practice is:

> Use the simplest mechanism that gives you the lifecycle hook your design actually needs.

# Extra exercises

Try extending the tutorial with these problems:

1. Add `staticmethod` and `classmethod` support to the `__init_subclass__` solution.
2. Add a `@no_trace` function decorator that makes one method opt out of metaclass tracing.
3. Modify `NoDuplicateMeta` so duplicate names are allowed only if they begin with `_temporary_`.
4. Make `RegistryMeta` support aliases.
5. Add unregister support to the decorator-based registry.
6. Make `InternMeta` reject unhashable constructor arguments with a clearer error.
7. Extend `NamedField` with a validator callable.
8. Add plugin priorities and return plugins sorted by priority.
9. Create two independent framework metaclasses and combine them cooperatively.
10. Compare a class decorator that returns a new subclass with one that mutates the original class.